# 02 — Baseline Model: TF-IDF + Logistic Regression

This notebook trains and evaluates the TF-IDF + LR baseline pipeline.

Steps:
1. Load preprocessed data
2. Build and train the pipeline
3. Evaluate on validation and test sets
4. Plot confusion matrix and ROC curve
5. Analyse feature importance (top coefficients)

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px

from config import set_seed
from src.data_loader import load_data, split_data
from src.baseline_model import build_pipeline, train_baseline, predict
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_curve

set_seed()
print('Setup complete.')

## 1. Load and Split Data

In [ ]:
df = load_data()
train_df, val_df, test_df = split_data(df)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

## 2. Train Baseline Pipeline

In [ ]:
pipeline = train_baseline(train_df, val_df)
print('Pipeline trained and saved.')

## 3. Test Set Evaluation

In [ ]:
y_test = test_df['label'].tolist()
preds, probs = predict(pipeline, test_df['clean_text'].tolist())

# For binary classification use positive-class probability
y_scores = probs[:, 1] if probs.shape[1] == 2 else probs.max(axis=1)

results = evaluate_model(y_test, preds, 'Baseline (TF-IDF + LR)', y_scores=y_scores)
pd.Series(results, name='Score').to_frame()

## 4. Confusion Matrix

In [ ]:
from IPython.display import Image
cm_path = plot_confusion_matrix(y_test, preds, 'Baseline (TF-IDF + LR)', labels=['Negative', 'Positive'])
Image(str(cm_path))

## 5. ROC Curve

In [ ]:
roc_path = plot_roc_curve(y_test, y_scores, 'Baseline (TF-IDF + LR)')
Image(str(roc_path))

## 6. Feature Importance — Top Coefficients

In [ ]:
import numpy as np

vectorizer = pipeline.named_steps['tfidf']
clf = pipeline.named_steps['clf']
feature_names = np.array(vectorizer.get_feature_names_out())

# Works for binary (1 class coef) and multi-class
coef = clf.coef_[0] if clf.coef_.shape[0] == 1 else clf.coef_[1]

top_pos_idx = coef.argsort()[-20:][::-1]
top_neg_idx = coef.argsort()[:20]

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=['Top Positive Features', 'Top Negative Features'])

fig.add_trace(go.Bar(
    x=coef[top_pos_idx], y=feature_names[top_pos_idx],
    orientation='h', marker_color='green', name='Positive'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=coef[top_neg_idx], y=feature_names[top_neg_idx],
    orientation='h', marker_color='red', name='Negative'
), row=1, col=2)

fig.update_layout(height=500, title_text='LR Coefficients — Most Influential Features', showlegend=False)
fig.show()